# Multimodal RAG: Macroeconomics PDF → submission CSV

PyMuPDF (text + page renders) · BGE-large-en · CLIP · ChromaDB · hybrid retrieval · Claude.

**Before you run:** enable a **GPU** runtime (optional but faster for CLIP) via *Runtime → Change runtime type*.

1. Run the install cell once.
2. Set your Anthropic API key and paths in the *Setup* cell. Upload `Document.pdf` and `Questions.csv` to the Colab session (or use Google Drive paths).
3. Run *ingest + QA* the first time. For a second run without rebuilding the index, set `NO_INGEST = True` in the setup cell and run the main cells again.


In [1]:
%%capture
# Colab: install dependencies (re-run if you get ImportError)
%pip install -q pymupdf torch sentence-transformers transformers rank-bm25 chromadb anthropic pandas numpy pillow


## Setup: API key, paths, run flags

If you use [Colab secrets](https://colab.research.google.com), add a secret named `ANTHROPIC_API_KEY`.


In [2]:
import os
import getpass
import importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if not os.environ.get("ANTHROPIC_API_KEY"):
    if IN_COLAB:
        try:
            from google.colab import userdata  # type: ignore
            os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        except Exception:
            os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")
    else:
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

os.environ["PDF_PATH"] = os.environ.get("PDF_PATH", "Document.pdf")
os.environ["QUESTIONS_CSV"] = os.environ.get("QUESTIONS_CSV", "Questions.csv")
os.environ["OUTPUT_CSV"] = os.environ.get("OUTPUT_CSV", "submission.csv")
os.environ["FIGURES_DIR"] = os.environ.get("FIGURES_DIR", "figures_rag")
os.environ["CHROMA_PERSIST"] = os.environ.get("CHROMA_PERSIST", "chroma_rag_db")

if IN_COLAB:
    from google.colab import files  # type: ignore
    # Uncomment to upload PDF + CSV from your machine (saves to /content)
    # uploaded = files.upload()

NO_INGEST = False
FORCE_INGEST = True
FIG_THRESHOLD = 0.30

print("PDF_PATH =", os.environ["PDF_PATH"])
print("QUESTIONS_CSV =", os.environ["QUESTIONS_CSV"])
print("NO_INGEST =", NO_INGEST)


PDF_PATH = Document.pdf
QUESTIONS_CSV = Questions.csv
NO_INGEST = False


In [3]:
from __future__ import annotations

import json
import logging
import os
import re
import shutil
import warnings
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import fitz
import torch
import anthropic
import chromadb
from PIL import Image
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import CLIPModel, CLIPProcessor

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIG
# =============================================================================


@dataclass
class Config:
    pdf_path: str = os.environ.get("PDF_PATH", "Document.pdf")
    questions_path: str = os.environ.get("QUESTIONS_CSV", "Questions.csv")
    # Leaderboard expects "submission.csv"; we write both names for safety
    output_path: str = os.environ.get("OUTPUT_CSV", "submission.csv")
    figures_dir: str = os.environ.get("FIGURES_DIR", "figures_rag")
    chroma_persist: str = os.environ.get("CHROMA_PERSIST", "chroma_rag_db")
    bge_model_name: str = "BAAI/bge-large-en"
    clip_model_name: str = "openai/clip-vit-base-patch32"
    claude_model: str = "claude-sonnet-4-20250514"
    claude_max_tokens: int = 1500
    claude_temperature: float = 0.0
    chunk_chars: int = 1200
    chunk_overlap_chars: int = 400
    page_render_scale: float = 2.0
    top_k_text_vector: int = 20
    top_k_fig_vector: int = 10
    final_text_chunks: int = 7
    final_figure_chunks: int = 3
    fig_combined_bge_weight: float = 0.5
    fig_combined_clip_weight: float = 0.5
    figure_similarity_threshold: float = 0.30
    hybrid_bge_weight: float = 0.6
    hybrid_bm25_weight: float = 0.4
    force_ingest: bool = False


# BAAI / MTEB-style prompts for bge-large-en
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
BGE_PASSAGE_PREFIX = "Represent this document for retrieval: "

# Match Figure 1, FIG. 2, Fig.3, Table 1, TABLE 1:, Figure 1-2, etc.
RE_FIG = re.compile(
    r"(?P<label>(?:\bFIGURE\b|\bFIG\.?\b|\bFig\.?\b|\bTable\b|\bTABLE\b))[\s:]*"
    r"(?P<num>\d+(?:[.\-–:]\d+)*)",
    re.IGNORECASE,
)

log = logging.getLogger("rag")


# =============================================================================
# TYPES
# =============================================================================


@dataclass
class TextChunk:
    chunk_id: str
    text: str
    page_number: int
    extra: Dict[str, Any] = field(default_factory=dict)


@dataclass
class FigureRecord:
    figure_id: str
    ftype: str
    number_token: str
    caption_text: str
    page_number: int
    image_path: str
    display_label: str


# =============================================================================
# LOGGING
# =============================================================================


def setup_logging() -> None:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        datefmt="%H:%M:%S",
    )


# =============================================================================
# PDF — TEXT
# =============================================================================


def _sweep(s: str) -> str:
    s = re.sub(r"\n{3,}", "\n\n", s)
    s = re.sub(r"[ \t]+", " ", s)
    return s.strip()


def extract_pages(pdf_path: str) -> List[Dict[str, Any]]:
    """Return every page (even with empty text) so figure scans still run on image-heavy pages."""
    doc = fitz.open(pdf_path)
    out = []
    for i in range(doc.page_count):
        t = _sweep(doc[i].get_text("text") or "")
        out.append({"page_number": i + 1, "text": t})
    doc.close()
    nz = sum(1 for p in out if p["text"])
    log.info("Read %d pages (%d with text) from %s", len(out), nz, pdf_path)
    return out


def _split_chars(text: str, page_number: int, start_idx: int, size: int, ov: int) -> List[TextChunk]:
    if not text or size <= 0:
        return []
    if ov >= size:
        ov = size // 4
    res: List[TextChunk] = []
    n = 0
    i = 0
    st = 0
    while st < len(text):
        ed = min(len(text), st + size)
        piece = text[st:ed].strip()
        if len(piece) > 30:
            cid = f"text_p{page_number}_{start_idx + n}"
            res.append(TextChunk(chunk_id=cid, text=piece, page_number=page_number))
            n += 1
        if ed == len(text):
            break
        st = ed - ov
        i += 1
    return res


def build_body_text_chunks(pages: List[Dict[str, Any]], cfg: Config) -> List[TextChunk]:
    all_c: List[TextChunk] = []
    g = 0
    for pg in pages:
        t = pg["text"]
        for ch in _split_chars(t, pg["page_number"], g, cfg.chunk_chars, cfg.chunk_overlap_chars):
            all_c.append(ch)
        g += 10_000
    log.info("Body text chunks: %d", len(all_c))
    return all_c


# =============================================================================
# PDF — FIGURES / TABLES
# =============================================================================


def _normalize_fig_type(m: re.Match) -> str:
    lab = m.group("label") or "Figure"
    t = lab.lower()
    if t.startswith("table"):
        return "table"
    return "figure"


def _span_key(ftype: str, num: str) -> str:
    return f"{ftype}:{num}"


def _find_captions_in_page(text: str) -> List[Tuple[str, str, int, int]]:
    """Return (ftype, number, start, end) for each match (no dedup here)."""
    hits: List[Tuple[str, str, int, int]] = []
    for m in RE_FIG.finditer(text):
        ftype = _normalize_fig_type(m)
        num = m.group("num")
        a, b = m.span()
        hits.append((ftype, num, a, b))
    hits.sort(key=lambda x: x[2])
    return hits


def _label_for(ftype: str, num: str) -> str:
    return f"Table {num}" if ftype == "table" else f"Figure {num}"


def extract_figures(
    pdf_path: str, pages: List[Dict[str, Any]], out_dir: str, scale: float
) -> Tuple[List[TextChunk], List[FigureRecord]]:
    """
    Build ONE record per unique (figure_type, number) across the whole document.
    - Render the page where the first mention occurs.
    - Merge surrounding text from every occurrence into a single caption blob
      (better recall for caption-based retrieval).
    """
    os.makedirs(out_dir, exist_ok=True)
    mat = fitz.Matrix(scale, scale)
    doc = fitz.open(pdf_path)

    # Group all mentions by (ftype, num)
    groups: Dict[Tuple[str, str], List[Tuple[int, int, int]]] = {}
    page_texts: Dict[int, str] = {}
    for pg in pages:
        pno = pg["page_number"]
        t = pg["text"]
        page_texts[pno] = t
        for ftype, num, a, b in _find_captions_in_page(t):
            groups.setdefault((ftype, num), []).append((pno, a, b))

    figs: List[FigureRecord] = []
    caps: List[TextChunk] = []
    for (ftype, num), mentions in sorted(groups.items()):
        # Pick the first mention's page for the rendered image
        first_pno, _, _ = mentions[0]
        page = doc.load_page(first_pno - 1)
        pix = page.get_pixmap(matrix=mat, alpha=False)
        png = pix.tobytes("png")
        safe = re.sub(r"[^\w.\-–]+", "_", num)
        fname = f"{'table' if ftype == 'table' else 'figure'}_{safe}.png"
        path = os.path.join(out_dir, fname)
        with open(path, "wb") as fh:
            fh.write(png)

        # Merge contexts from ALL mentions for richer caption text
        ctx_parts: List[str] = []
        for (pno, a, b) in mentions:
            t = page_texts.get(pno, "")
            piece = _sweep(t[max(0, a - 400) : min(len(t), b + 800)])
            if piece and piece not in ctx_parts:
                ctx_parts.append(piece)
        dlab = _label_for(ftype, num)
        caption = f"{dlab}. " + "  \n".join(ctx_parts)
        fid = f"{'tbl' if ftype == 'table' else 'fig'}_{num}"

        figs.append(
            FigureRecord(
                figure_id=fid,
                ftype=ftype,
                number_token=num,
                caption_text=caption,
                page_number=first_pno,
                image_path=path,
                display_label=dlab,
            )
        )
        caps.append(
            TextChunk(
                chunk_id=f"caption_{fid}",
                text=caption,
                page_number=first_pno,
                extra={"linked_figure": fid, "label": dlab},
            )
        )
        log.info("Saved %s | %s (mentions=%d)", fname, dlab, len(mentions))

    doc.close()
    log.info("Unique figures/tables: %d (caption chunks: %d)", len(figs), len(caps))
    return caps, figs


# =============================================================================
# EMBEDDINGS
# =============================================================================


def load_bge(name: str) -> SentenceTransformer:
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    log.info("Loading BGE: %s on %s", name, dev)
    return SentenceTransformer(name, device=dev)


def bge_dim(m: SentenceTransformer) -> int:
    return int(m.get_sentence_embedding_dimension())


def bge_passages(m: SentenceTransformer, texts: List[str], bs: int = 64) -> np.ndarray:
    if not texts:
        return np.zeros((0, bge_dim(m)), np.float32)
    batch = [BGE_PASSAGE_PREFIX + t for t in texts]
    e = m.encode(
        batch,
        batch_size=bs,
        convert_to_numpy=True,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    return e.astype(np.float32)


def bge_query_vec(m: SentenceTransformer, q: str) -> np.ndarray:
    t = BGE_QUERY_PREFIX + q
    e = m.encode([t], convert_to_numpy=True, normalize_embeddings=True)
    return e.astype(np.float32)


def load_clip(name: str, dev: Optional[str] = None) -> Tuple[CLIPModel, CLIPProcessor, str]:
    d = dev or ("cuda" if torch.cuda.is_available() else "cpu")
    log.info("Loading CLIP %s on %s", name, d)
    model = CLIPModel.from_pretrained(name).to(d)
    proc = CLIPProcessor.from_pretrained(name)
    model.eval()
    return model, proc, d


def _clip_to_device(
    ins: Any, dev: str
) -> Dict[str, torch.Tensor]:
    """Move processor outputs to device without spreading unknown keys into the model."""
    if hasattr(ins, "to"):
        ins = ins.to(dev)
    out: Dict[str, torch.Tensor] = {}
    for k, v in ins.items():
        if torch.is_tensor(v):
            out[k] = v.to(dev)
    return out


def _clip_image_embeds(model: CLIPModel, pixel_values: torch.Tensor) -> torch.Tensor:
    """
    Image-side CLIP embeddings only (no text tower). Avoids CLIPModel.forward() on recent
    transformers, which may require both modalities and raise "You have to specify input_ids".
    Mirrors get_image_features: visual_projection(vision_model(...).pooler_output).
    """
    vision_out = model.vision_model(pixel_values=pixel_values, return_dict=True)
    pooled = getattr(vision_out, "pooler_output", None)
    if pooled is None:
        pooled = vision_out[1]
    f = model.visual_projection(pooled)
    if not torch.is_tensor(f):
        raise RuntimeError("visual_projection did not return a tensor")
    return f


def _clip_text_embeds(
    model: CLIPModel, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor]
) -> torch.Tensor:
    """Text-side only; mirrors get_text_features without full model forward."""
    text_out = model.text_model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        return_dict=True,
    )
    pooled = getattr(text_out, "pooler_output", None)
    if pooled is None:
        pooled = text_out[1]
    f = model.text_projection(pooled)
    if not torch.is_tensor(f):
        raise RuntimeError("text_projection did not return a tensor")
    return f


@torch.inference_mode()
def clip_imgs(
    model: CLIPModel, proc: CLIPProcessor, dev: str, paths: List[str], bs: int = 32
) -> np.ndarray:
    if not paths:
        return np.zeros((0, 512), np.float32)
    outs: List[np.ndarray] = []
    for i in range(0, len(paths), bs):
        batch = [Image.open(p).convert("RGB") for p in paths[i : i + bs]]
        ins = _clip_to_device(
            proc(images=batch, return_tensors="pt", padding=True), dev
        )
        f = _clip_image_embeds(model, ins["pixel_values"])
        f = f / (f.norm(dim=-1, keepdim=True) + 1e-10)
        outs.append(f.float().cpu().numpy())
    return np.vstack(outs).astype(np.float32) if outs else np.zeros((0, 512), np.float32)


@torch.inference_mode()
def clip_txt(
    model: CLIPModel, proc: CLIPProcessor, dev: str, texts: List[str], bs: int = 32
) -> np.ndarray:
    if not texts:
        return np.zeros((0, 512), np.float32)
    outs: List[np.ndarray] = []
    for i in range(0, len(texts), bs):
        batch = texts[i : i + bs]
        ins = _clip_to_device(
            proc(
                text=batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=77,
            ),
            dev,
        )
        f = _clip_text_embeds(
            model,
            ins["input_ids"],
            ins.get("attention_mask"),
        )
        f = f / (f.norm(dim=-1, keepdim=True) + 1e-10)
        outs.append(f.float().cpu().numpy())
    return np.vstack(outs).astype(np.float32) if outs else np.zeros((0, 512), np.float32)


# =============================================================================
# PERSIST: NPZ (CLIP image vectors; Chroma has BGE caption)
# =============================================================================


def save_emb_npz(
    path: str, ids: List[str], bge_mat: np.ndarray, clip_mat: np.ndarray, bge_d: int, clip_d: int
) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    np.savez(
        path,
        figure_ids=np.array(ids, dtype=object),
        bge_caption=bge_mat,
        clip_image=clip_mat,
        bge_dim=bge_d,
        clip_dim=clip_d,
    )
    with open(path.replace(".npz", ".json"), "w", encoding="utf-8") as f:
        json.dump(
            {
                "figure_ids": ids,
                "bge_dim": bge_d,
                "clip_dim": clip_d,
            },
            f,
        )
    log.info("Saved %s (n=%d)", path, len(ids))


def load_emb_npz(path: str) -> Tuple[List[str], np.ndarray, np.ndarray, int, int]:
    z = np.load(path, allow_pickle=True)
    ids = [str(x) for x in z["figure_ids"].tolist()]
    b = z["bge_caption"] if "bge_caption" in z else None
    c = z["clip_image"] if "clip_image" in z else None
    bd = int(z["bge_dim"]) if "bge_dim" in z else (b.shape[1] if b is not None and len(b) else 1024)
    cd = int(z["clip_dim"]) if "clip_dim" in z else (c.shape[1] if c is not None and len(c) else 512)
    if b is None:
        b = np.zeros((0, bd), np.float32)
    if c is None:
        c = np.zeros((0, cd), np.float32)
    return ids, b, c, bd, cd


# =============================================================================
# CHROMA INGEST
# =============================================================================


def _reset_chroma(path: str) -> chromadb.api.client.Client:
    if os.path.isdir(path):
        shutil.rmtree(path, ignore_errors=True)
    os.makedirs(path, exist_ok=True)
    client = chromadb.PersistentClient(path=path)
    for col in client.list_collections():
        client.delete_collection(col.name)
    return client


def run_full_ingest(cfg: Config) -> Tuple[
    Any,
    Any,
    Any,
    str,
    chromadb.api.client.Client,
    Any,
    Any,
    List[str],
    np.ndarray,
    np.ndarray,
]:
    """Build figures, Chroma, and embeddings cache. Wipes `chroma_persist` for a clean add."""
    if not os.path.isfile(cfg.pdf_path):
        raise FileNotFoundError(f"PDF not found: {cfg.pdf_path}")

    pages = extract_pages(cfg.pdf_path)
    body = build_body_text_chunks(pages, cfg)
    cap_txt, figs = extract_figures(cfg.pdf_path, pages, cfg.figures_dir, cfg.page_render_scale)
    all_text = body + cap_txt

    bge = load_bge(cfg.bge_model_name)
    bdim = bge_dim(bge)
    clip_m, clip_p, dev = load_clip(cfg.clip_model_name)
    t_emb = bge_passages(bge, [c.text for c in all_text])
    caps = [f.caption_text for f in figs]
    pths = [f.image_path for f in figs]
    cap_bge = bge_passages(bge, caps) if caps else np.zeros((0, bdim), np.float32)
    clip_m_img = clip_imgs(clip_m, clip_p, dev, pths) if pths else np.zeros((0, 512), np.float32)
    fids = [f.figure_id for f in figs]

    npz_path = os.path.join(cfg.figures_dir, "embeddings_cache.npz")
    save_emb_npz(npz_path, fids, cap_bge, clip_m_img, bdim, 512)

    client = _reset_chroma(cfg.chroma_persist)
    tcol = client.create_collection("text_chunks", metadata={"hnsw:space": "cosine"})
    fcol = client.create_collection("figures_tables", metadata={"hnsw:space": "cosine"})

    t_ids = [c.chunk_id for c in all_text]
    t_meta = [
        {
            "page_number": c.page_number,
            "chunk_id": c.chunk_id,
            "linked_figure": str(c.extra.get("linked_figure", "")),
        }
        for c in all_text
    ]
    for s in range(0, len(t_ids), 64):
        tcol.add(
            ids=t_ids[s : s + 64],
            embeddings=[v.tolist() for v in t_emb[s : s + 64]],
            documents=[c.text for c in all_text][s : s + 64],
            metadatas=t_meta[s : s + 64],
        )
    f_meta = [
        {
            "figure_number": f.number_token,
            "figure_type": f.ftype,
            "caption": f.caption_text[:2000],
            "page_number": f.page_number,
            "image_path": f.image_path,
            "display_label": f.display_label,
        }
        for f in figs
    ]
    for s in range(0, len(fids), 32):
        fcol.add(
            ids=fids[s : s + 32],
            embeddings=[v.tolist() for v in cap_bge[s : s + 32]],
            documents=caps[s : s + 32],
            metadatas=f_meta[s : s + 32],
        )
    log.info("Chroma: text_chunks=%d, figures_tables=%d", len(t_ids), len(fids))
    return tcol, fcol, bge, dev, client, clip_m, clip_p, fids, cap_bge, clip_m_img


# =============================================================================
# RETRIEVAL
# =============================================================================


def tok_bm25(s: str) -> List[str]:
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", s.lower())


def ret_text_hybrid(
    q: str, bge: SentenceTransformer, col, cfg: Config,
) -> Tuple[List[Tuple[Dict[str, Any], float]], List[str]]:
    """Hybrid retrieval: BGE cosine similarity + BM25 re-rank."""
    ntot = int(col.count())
    if ntot == 0:
        return [], []
    qv = bge_query_vec(bge, q)
    nres = min(cfg.top_k_text_vector, ntot)
    r = col.query(
        query_embeddings=qv.tolist(),
        n_results=nres,
        include=["documents", "metadatas", "distances"],
    )
    ids = (r.get("ids") or [[]])[0]
    docs = (r.get("documents") or [[]])[0]
    mets = (r.get("metadatas") or [[]])[0]
    dists = (r.get("distances") or [[]])[0]
    packed: List[Tuple[Dict[str, Any], float]] = []
    for i, d in enumerate(docs):
        dist = float(dists[i]) if i < len(dists) else 0.0
        sim = max(0.0, 1.0 - dist)
        row = {
            "id": ids[i],
            "text": d,
            "metadata": mets[i] if i < len(mets) else {},
        }
        packed.append((row, sim))
    if not packed:
        return [], []

    bge_sims = np.array([s for _, s in packed], dtype=np.float32)

    corpus = [p[0]["text"] for p in packed]
    bm = BM25Okapi([tok_bm25(t) for t in corpus])
    bm25_raw = bm.get_scores(tok_bm25(q))
    bm25_max = bm25_raw.max() if bm25_raw.max() > 0 else 1.0
    bm25_norm = bm25_raw / bm25_max

    hybrid = cfg.hybrid_bge_weight * bge_sims + cfg.hybrid_bm25_weight * bm25_norm

    order = list(np.argsort(hybrid)[::-1])
    out = [(packed[i][0], float(hybrid[i])) for i in order[: cfg.final_text_chunks]]
    tids = [o[0]["id"] for o in out]
    return out, tids


def ret_figures(
    q: str,
    bge: SentenceTransformer,
    clip_m: CLIPModel,
    clip_p: CLIPProcessor,
    dev: str,
    fcol: Any,
    fids: List[str],
    bge_mat: np.ndarray,
    clip_mat: np.ndarray,
    cfg: Config,
) -> List[Tuple[Dict[str, Any], float]]:
    if not fids or bge_mat.size == 0 or clip_mat.size == 0:
        return []
    id2j = {x: j for j, x in enumerate(fids)}
    qb = bge_query_vec(bge, q)[0]
    qc = clip_txt(clip_m, clip_p, dev, [q])[0]
    nf = int(fcol.count())
    if nf == 0:
        return []
    nres = min(cfg.top_k_fig_vector, nf)
    r = fcol.query(
        query_embeddings=bge_query_vec(bge, q).tolist(),
        n_results=nres,
        include=["metadatas"],
    )
    fids_q = (r.get("ids") or [[]])[0]
    rmet = (r.get("metadatas") or [[]])[0]
    res: List[Tuple[Dict[str, Any], float]] = []
    for idx, rid in enumerate(fids_q):
        if rid not in id2j:
            continue
        j = id2j[rid]
        b = float(np.dot(qb, bge_mat[j]))
        c = float(np.dot(qc, clip_mat[j]))
        s = cfg.fig_combined_bge_weight * b + cfg.fig_combined_clip_weight * c
        meta = dict(rmet[idx] if idx < len(rmet) and rmet[idx] else {})
        res.append((meta, s))
    res.sort(key=lambda x: -x[1])
    return res[: cfg.final_figure_chunks]


# =============================================================================
# CLAUDE
# =============================================================================


def _make_query_variants(q: str) -> List[str]:
    """Generate multiple query reformulations for multi-query retrieval."""
    variants = []
    q_lower = q.lower()
    formal = q
    replacements = [
        ("what sparked", "what caused"),
        ("how bad", "how severe was the impact"),
        ("get hit", "affected"),
        ("tank", "decline in"),
        ("kicked in", "began"),
        ("messing it up", "distorting the measurement"),
        ("keep growing strong", "maintain high growth"),
        ("out of work", "unemployed"),
        ("in sync", "correlated"),
        ("struggled with jobs", "experienced high unemployment"),
        ("how bad did", "what was the severity of"),
        ("grew faster", "had higher growth rates"),
    ]
    for old, new in replacements:
        if old in q_lower:
            formal = re.sub(re.escape(old), new, formal, flags=re.IGNORECASE)
    if formal != q:
        variants.append(formal)
    keywords = re.findall(r'\b[A-Z][a-z]+\b|\b\d{4}\b|\b[A-Z]{2,}\b', q)
    topic_words = [w for w in q.split() if len(w) > 4 and w.lower() not in
                   {"about", "which", "where", "there", "their", "these", "those", "should", "would", "could"}]
    if keywords or topic_words:
        kw_query = " ".join(keywords + topic_words[:5])
        if kw_query != q:
            variants.append(kw_query)
    return variants


def _augment_query(q: str) -> str:
    """
    Expand colloquial question phrasing with formal economics keywords
    to improve BGE retrieval recall on academic text.
    """
    expansions: Dict[str, List[str]] = {
        "2008": ["financial crisis", "Great Recession", "subprime", "credit crunch"],
        "2009": ["recession", "economic downturn", "contraction", "output decline"],
        "crisis": ["financial crisis", "2008", "Great Recession", "subprime", "credit crisis"],
        "unemployment": ["unemployment rate", "labor market", "jobless", "employment"],
        "GDP": ["gross domestic product", "real GDP", "output", "national income"],
        "growth": ["economic growth", "GDP growth", "growth rate", "output growth"],
        "grew": ["economic growth", "GDP growth", "growth rate", "output growth"],
        "stock market": ["stock prices", "equity markets", "S&P 500", "Dow Jones"],
        "stock": ["stock prices", "equity markets", "S&P 500", "Dow Jones", "stock market"],
        "housing": ["housing prices", "house prices", "real estate", "housing bubble", "housing wealth"],
        "mortgage": ["mortgage-backed securities", "MBS", "subprime mortgage", "securitization"],
        "inflation": ["consumer price index", "CPI", "price level", "GDP deflator"],
        "prices": ["consumer price index", "CPI", "price level", "GDP deflator", "inflation"],
        "recession": ["recession", "contraction", "negative growth", "downturn", "decline in output"],
        "China": ["China", "Chinese economy", "emerging market", "emerging economies"],
        "Europe": ["Europe", "euro area", "European Union", "eurozone", "European unemployment"],
        "consumer": ["consumption", "consumer spending", "household spending", "consumer confidence"],
        "consumption": ["consumer spending", "household spending", "consumption expenditure"],
        "advanced": ["advanced economies", "developed countries", "United States", "euro area"],
        "emerging": ["emerging economies", "developing countries", "China", "India", "emerging market"],
        "economy": ["economic output", "GDP", "macroeconomic", "aggregate output"],
        "economic": ["macroeconomic", "GDP", "output", "aggregate"],
        "measure": ["nominal GDP", "real GDP", "GDP deflator", "CPI", "price index"],
        "work": ["employment", "labor market", "unemployment", "jobs"],
    }
    extra_terms: List[str] = []
    ql = q.lower()
    for trigger, terms in expansions.items():
        if trigger.lower() in ql:
            for t in terms:
                if t.lower() not in ql:
                    extra_terms.append(t)
    if not extra_terms:
        return q
    seen = set()
    deduped = [t for t in extra_terms if t not in seen and not seen.add(t)]
    return q + " " + " ".join(deduped[:10])


def build_user_block(
    q: str, tpack: List[Tuple[Dict[str, Any], float]], fpack: List[Tuple[Dict[str, Any], float]]
) -> str:
    lines = [f"QUESTION:\n{q}\n"]

    lines.append("\n--- RETRIEVED TEXT PASSAGES ---\n")
    for i, (row, sim) in enumerate(tpack, 1):
        md = row.get("metadata") or {}
        lines.append(
            f"[Passage {i} | page {md.get('page_number', '')}]\n{row['text']}\n\n"
        )
    if not tpack:
        lines.append("(none)\n")

    lines.append("\n--- RETRIEVED FIGURE / TABLE CAPTIONS ---\n")
    for i, (m, s) in enumerate(fpack, 1):
        cap = (m.get("caption") or "")[:1500]
        label = m.get("display_label", "")
        lines.append(
            f"[{label} | page {m.get('page_number', '')}]\n{cap}\n\n"
        )
    if not fpack:
        lines.append("(none)\n")

    avail_figs = ", ".join(m.get("display_label", "") for m, _ in fpack if m.get("display_label"))
    if not avail_figs:
        avail_figs = "(none retrieved)"

    lines.append(
        "\nAnswer the question using ONLY the passages and figure captions above. "
        "Prefer verbatim phrases from the text. Include numbers, dates, and percentages exactly.\n\n"
        f"AVAILABLE figure/table IDs: {avail_figs}\n\n"
        "Respond in this EXACT XML format:\n"
        "<answer>your 2-4 sentence answer</answer>\n"
        "<refs>one ID like \"Figure 1-2\" or \"Table 1-1\", or \"0\"</refs>\n"
    )
    return "".join(lines)


def claude_ans(cfg: Config, system: str, user: str) -> str:
    key = os.environ.get("ANTHROPIC_API_KEY", "")
    if not key:
        raise RuntimeError("Set ANTHROPIC_API_KEY in the environment.")
    client = anthropic.Anthropic(api_key=key)
    m = client.messages.create(
        model=cfg.claude_model,
        max_tokens=cfg.claude_max_tokens,
        temperature=cfg.claude_temperature,
        system=system,
        messages=[{"role": "user", "content": user}],
    )
    if not m.content or not m.content[0].text:
        return ""
    return m.content[0].text.strip()


RE_XML_ANSWER = re.compile(r"<answer>(.*?)</answer>", re.DOTALL | re.IGNORECASE)
RE_XML_REFS = re.compile(r"<refs>(.*?)</refs>", re.DOTALL | re.IGNORECASE)
RE_FIGURE_REF_LINE = re.compile(r"FIGURE_REF:\s*(.+)", re.IGNORECASE)


def parse_claude_response(raw: str) -> Tuple[str, str]:
    """Parse Claude XML (<answer>/<refs>) or legacy FIGURE_REF: format."""
    fig_ref = "0"

    ans_m = RE_XML_ANSWER.search(raw)
    refs_m = RE_XML_REFS.search(raw)

    if ans_m:
        answer = ans_m.group(1).strip()
    else:
        answer = raw.strip()

    ref_raw = ""
    if refs_m:
        ref_raw = refs_m.group(1).strip()
    else:
        for line in raw.strip().split("\n"):
            m = RE_FIGURE_REF_LINE.match(line.strip())
            if m:
                ref_raw = m.group(1).strip()
                break

    if ref_raw and ref_raw not in ("0", "none", "n/a", "no"):
        fm = RE_FIG.search(ref_raw)
        if fm:
            ftype = _normalize_fig_type(fm)
            num = fm.group("num")
            fig_ref = _label_for(ftype, num)

    if not answer:
        answer = raw.strip()

    return answer, fig_ref


# =============================================================================
# CSV
# =============================================================================


def load_questions(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    # Detect which column holds the actual question text
    if "question" in df.columns:
        qtext = df["question"].astype(str)
    elif "question_id" in df.columns and df["question_id"].dtype == object:
        # Questions.csv: "question_id" column actually contains the question text
        qtext = df["question_id"].astype(str)
    else:
        raise ValueError("Need a `question` or text-valued `question_id` column in the CSV")

    # Unique row id (used as submission `id`)
    row_ids = df["id"].astype(int) if "id" in df.columns else list(range(1, len(df) + 1))

    # conversation_id — use explicit column, else fall back to row id
    if "conversation_id" in df.columns:
        conv = df["conversation_id"].astype(str)
    else:
        conv = pd.Series([str(x) for x in row_ids])

    # question_id (numeric) — use q_id if present, else row id
    if "q_id" in df.columns:
        qid_col = df["q_id"]
    else:
        qid_col = row_ids

    return pd.DataFrame(
        {
            "submission_id": row_ids,
            "conversation_id": conv.values,
            "question_id": qid_col,
            "question_text": qtext.values,
        }
    )


# =============================================================================
# MAIN FLOW
# =============================================================================


def run(cfg: Config, no_ingest: bool) -> None:
    if no_ingest:
        npz = os.path.join(cfg.figures_dir, "embeddings_cache.npz")
        if not (os.path.isdir(cfg.chroma_persist) and os.path.isfile(npz)):
            raise FileNotFoundError("Run once without --no-ingest, or set FIGURES_DIR to an indexed folder.")
        client = chromadb.PersistentClient(path=cfg.chroma_persist)
        tcol = client.get_collection("text_chunks")
        fcol = client.get_collection("figures_tables")
        bge = load_bge(cfg.bge_model_name)
        clip_m, clip_p, dev = load_clip(cfg.clip_model_name)
        fids, bmat, cmat, _, _ = load_emb_npz(npz)
    else:
        if cfg.force_ingest and os.path.isdir(cfg.figures_dir):
            for fn in os.listdir(cfg.figures_dir):
                p = os.path.join(cfg.figures_dir, fn)
                if os.path.isfile(p) and (fn.endswith(".png") or fn.endswith(".npz") or fn.endswith(".json")):
                    try:
                        os.remove(p)
                    except OSError:
                        pass
        tcol, fcol, bge, dev, client, clip_m, clip_p, fids, bmat, cmat = run_full_ingest(cfg)

    qdf = load_questions(cfg.questions_path)
    sysm = (
        "You are a precise economics teaching assistant.\n"
        "Answer STRICTLY from the provided text excerpts and images.\n"
        "Use wording as close to the original document as possible — quote or "
        "paraphrase the source text directly. Include specific numbers, dates, "
        "percentages, and country names exactly as stated in the evidence.\n\n"
        "Answer rules:\n"
        "- Keep the answer to 2-4 sentences.\n"
        "- Prefer VERBATIM phrases from the text evidence over your own phrasing.\n"
        "- If the evidence does not clearly support an answer, reply: Not found in document.\n\n"
        "Figure/Table CITATION DECISION (critical):\n\n"
        "STEP 1. Check whether the text evidence explicitly mentions a Figure or "
        "Table by name (e.g. 'as shown in Figure 1-2', 'Table 1-4 shows', "
        "'the numbers in Table 1-3'). If YES, cite that Figure/Table.\n\n"
        "STEP 2. If no Figure/Table is explicitly named in the text evidence, "
        "classify the question:\n"
        "  A) DEFINITION / CAUSE / MECHANISM / CONCEPT\n"
        "     (e.g. 'What is X?', 'What sparked Y?', 'Why should we worry...', "
        "'What role did X play?', 'What happens when X?')\n"
        "     -> refs = \"0\". These are answered from text, not charts.\n\n"
        "  B) DATA / TREND / COMPARISON / SPECIFIC NUMBER\n"
        "     (e.g. 'How much did X grow?', 'What happened to X after 2008?', "
        "'How bad was it in Y?', 'Compare X vs Y', 'Did X rise or fall?', "
        "'Which group grew faster?')\n"
        "     -> cite EXACTLY ONE figure or table whose IMAGE most directly "
        "shows the quantity asked about.\n\n"
        "STEP 3. When choosing which figure/table to cite:\n"
        "  - Pick the ONE whose CAPTION explicitly describes the quantity/entity.\n"
        "  - Use the IMAGE to confirm the axes/labels match the question.\n"
        "  - Prefer a Table when the question asks for specific numbers across "
        "years/countries. Prefer a Figure when it asks about a visual trend.\n"
        "  - Only cite IDs from the AVAILABLE list. Never fabricate.\n\n"
        "Output EXACTLY this XML, nothing else:\n"
        "<answer>your 2-4 sentence answer</answer>\n"
        "<refs>one ID like \"Figure 1-2\" or \"Table 1-1\", or \"0\"</refs>\n"
    )
    out_rows: List[Dict[str, Any]] = []
    for _, row in qdf.iterrows():
        q = str(row["question_text"])
        log.info("Q: %s", q[:180])

        q_aug = _augment_query(q)
        tpack, tids = ret_text_hybrid(q_aug, bge, tcol, cfg)
        fpack = ret_figures(
            q_aug, bge, clip_m, clip_p, dev, fcol, fids, bmat, cmat, cfg
        )
        u = build_user_block(q, tpack, fpack)
        log.info("Text (hybrid BGE+BM25 top-%d) ids: %s", cfg.top_k_text_vector, tids)
        log.info("Figures: %s", [(a.get("display_label"), f"{b:.3f}") for a, b in fpack])
        raw_ans = claude_ans(cfg, sysm, u)

        ans, figs_str = parse_claude_response(raw_ans)
        log.info("Figure ref: %s", figs_str)

        out_rows.append(
            {
                "id": int(row["submission_id"]),
                "conversation_id": str(row["conversation_id"]),
                "question_id": int(row["question_id"]),
                "answer": ans,
                "figure_references": figs_str,
            }
        )
    df_out = pd.DataFrame(out_rows)
    df_out.to_csv(cfg.output_path, index=False)
    # Mirror file: leaderboard expects the spec-named "submission.csv" but our
    # default config also accepts "submissions.csv" — write both for safety.
    base_dir = os.path.dirname(os.path.abspath(cfg.output_path)) or "."
    for alt in ("submission.csv", "submissions.csv"):
        alt_path = os.path.join(base_dir, alt)
        if os.path.abspath(alt_path) != os.path.abspath(cfg.output_path):
            df_out.to_csv(alt_path, index=False)
            log.info("Wrote mirror: %s", alt_path)
    log.info("Wrote %s (%d rows)", cfg.output_path, len(out_rows))


In [4]:
setup_logging()
c = Config(
    force_ingest=FORCE_INGEST,
    figure_similarity_threshold=FIG_THRESHOLD,
)
run(c, no_ingest=NO_INGEST)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

## Preview & Download `submission.csv`

In [5]:
import pandas as pd

sub = pd.read_csv("submission.csv")
print(f"submission.csv — {len(sub)} rows, columns: {list(sub.columns)}")
display(sub)

if IN_COLAB:
    from google.colab import files  # type: ignore
    files.download("submission.csv")

submission.csv — 15 rows, columns: ['id', 'conversation_id', 'question_id', 'answer', 'figure_references']


,id,conversation_id,question_id,answer,figure_references
0,1,1,1,The global economic crisis around 2008 was spa...,0
1,2,2,2,Economists care about unemployment for two rea...,0
2,3,3,3,Economists measure economic growth without pri...,0
3,4,4,4,The world economy was severely hit during the ...,Table 1-1
4,5,5,5,"As shown in Figure 1-2, ""the increase in the u...",Figure 1-2
5,6,6,6,"According to Table 1-4, China's output growth ...",Table 1-4
6,7,7,7,The 2008 crisis tanked stock markets everywher...,Figure 1-1
7,8,8,8,"No, fast economic growth does not always mean ...",Figure 2-5
8,9,9,9,"No, consumer prices and overall economic price...",Figure 2-4
9,10,10,10,Even during the pre-crisis period from 2000 to...,Table 1-3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>